In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/nightfall'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
print(f"Drive mounted. Nightfall root: {DRIVE_ROOT}")

In [ ]:
!rm -rf /content/Nightfall
os.chdir('/content')

In [ ]:
!git clone https://github.com/JeremiahAdebayo/nightfall.git /content/Nightfall
%cd /content/Nightfall/

In [ ]:
!pip install -e ".[data]" --quiet
!pip install scipy scikit-image scikit-learn --quiet

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type")

In [ ]:
MVTEC_DIR = f'{DRIVE_ROOT}/data/mvtec_ad'
os.makedirs(MVTEC_DIR, exist_ok=True)
print(f"Dataset will be auto-downloaded per-category into: {MVTEC_DIR}")

In [ ]:
'''!python eval/train.py \
    --data-root {MVTEC_DIR} \
    --output-dir {DRIVE_ROOT}/checkpoints \
    --categories bottle'''

In [ ]:
'''!python eval/train.py \
    --data-root {MVTEC_DIR} \
    --output-dir {DRIVE_ROOT}/checkpoints'''

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img_path = f"{MVTEC_DIR}/bottle/test/broken_large/004.png"
mask_path = f"{MVTEC_DIR}/bottle/ground_truth/broken_large/004_mask.png"

img = Image.open(img_path)
mask = Image.open(mask_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("test image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("ground truth mask")
plt.show()

In [ ]:
!python eval/run_eval.py \
    --data-root {MVTEC_DIR} \
    --checkpoint-dir {DRIVE_ROOT}/checkpoints

In [ ]:
pip install numpy==1.26.4

In [ ]:
pip install --force-reinstall onnxruntime

In [ ]:
"""
Export PatchFeatureExtractor to ONNX, then verify numerical correctness
against the original PyTorch model on the same input.

Verification is not optional here: PatchFeatureExtractor.forward() reads
patch features back out of self._features, a plain Python dict populated
as a side effect of forward hooks on layer2/layer3. ONNX's tracer records
tensor operations executed during a forward pass, but whether it reliably
captures a hook-populated dict read as a genuine data dependency in the
exported graph is a known sharp edge, not something to assume works
without checking. If verification fails or is close-but-not-exact, that's
real information about whether this export path is trustworthy -- not
something to paper over.

Usage:
    !python scripts/export_onnx.py --output-dir {DRIVE_ROOT}/onnx
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import torch

from core.feature_extractor import PatchFeatureExtractor


def export_and_verify(output_dir: Path, input_size: int = 224) -> bool:
    output_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = output_dir / "feature_extractor.onnx"

    model = PatchFeatureExtractor()
    model.eval()

    dummy_input = torch.randn(1, 3, input_size, input_size)

    # Get the real PyTorch output first, on this exact input, so we have
    # a ground truth to compare the exported graph against.
    with torch.no_grad():
        torch_output = model(dummy_input)

    print(f"PyTorch output shape: {torch_output.shape}")

    print(f"Exporting to {onnx_path}...")
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_path),
        input_names=["image"],
        output_names=["patch_features"],
        dynamic_axes={
            "image": {0: "batch_size"},
            "patch_features": {0: "batch_size"},
        },
        opset_version=17,
    )
    print(f"Export completed, file size: {onnx_path.stat().st_size / 1e6:.1f} MB")

    # Verification: load the exported graph and run it via onnxruntime,
    # compare against the PyTorch output on the SAME input tensor.
    import onnxruntime as ort

    session = ort.InferenceSession(str(onnx_path))
    onnx_output = session.run(
        ["patch_features"], {"image": dummy_input.numpy()}
    )[0]

    torch_output_np = torch_output.numpy()

    if onnx_output.shape != torch_output_np.shape:
        print(
            f"[FAIL] Shape mismatch: ONNX {onnx_output.shape} vs "
            f"PyTorch {torch_output_np.shape}. This strongly suggests the "
            f"hook-based feature capture did NOT export correctly -- the "
            f"graph is structurally wrong, not just numerically imprecise."
        )
        return False

    max_abs_diff = np.abs(onnx_output - torch_output_np).max()
    mean_abs_diff = np.abs(onnx_output - torch_output_np).mean()
    print(f"Max absolute difference: {max_abs_diff:.6e}")
    print(f"Mean absolute difference: {mean_abs_diff:.6e}")

    # A small numerical tolerance is expected and fine (different backend
    # kernels, floating point op ordering) -- this is NOT the same
    # tolerance question as the reweighting bug we caught earlier, since
    # here we're checking backend equivalence, not algorithm correctness.
    # A large or NaN difference means the export is structurally broken,
    # most likely due to the hook-based feature capture not being traced
    # as a real data dependency.
    tolerance = 1e-3
    if max_abs_diff > tolerance:
        print(
            f"[FAIL] Max difference {max_abs_diff:.6e} exceeds tolerance "
            f"{tolerance:.0e}. Do NOT trust this ONNX export -- the hook-based "
            f"forward() likely did not export correctly. Consider rewriting "
            f"forward() to return the fused feature map directly without "
            f"relying on hook-populated instance state, then re-export."
        )
        return False

    print(f"[PASS] ONNX export verified within tolerance ({tolerance:.0e}).")
    return True


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--input-size", type=int, default=224)
    args = parser.parse_args()
from pathlib import Path

success = export_and_verify(Path(f"{DRIVE_ROOT}/onnx"), 224)
if not success:
    raise SystemExit(
        "ONNX export failed verification -- see output above for details. "
        "Do not proceed to quantization with an unverified export."
    )


if __name__ == "__main__":
    main()

In [ ]:
import onnx

model = onnx.load(f"{DRIVE_ROOT}/onnx/feature_extractor.onnx")

# Count actual weight tensors (initializers) and their total size
total_params = 0
for initializer in model.graph.initializer:
    shape = [d for d in initializer.dims]
    num_elements = 1
    for d in shape:
        num_elements *= d
    total_params += num_elements

print(f"Number of weight tensors (initializers): {len(model.graph.initializer)}")
print(f"Total parameter count: {total_params:,}")
print(f"Number of graph nodes (ops): {len(model.graph.node)}")

# List the first few and last few node op types, to see what's actually in the graph
op_types = [node.op_type for node in model.graph.node]
print(f"\nFirst 10 ops: {op_types[:10]}")
print(f"Last 10 ops: {op_types[-10:]}")

In [ ]:
import os
file_size_mb = os.path.getsize(f"{DRIVE_ROOT}/onnx/feature_extractor.onnx") / 1e6
expected_fp32_mb = (24_842_754 * 4) / 1e6
print(f"Actual file size: {file_size_mb:.1f} MB")
print(f"Expected size if fp32 uncompressed: {expected_fp32_mb:.1f} MB")

In [ ]:
import os
onnx_dir = f"{DRIVE_ROOT}/onnx"
for f in os.listdir(onnx_dir):
    full_path = os.path.join(onnx_dir, f)
    size_mb = os.path.getsize(full_path) / 1e6
    print(f"{f}: {size_mb:.2f} MB")

In [ ]:
"""
INT8 post-training quantization of Nightfall's exported feature
extractor, using ONNX Runtime's dynamic quantization (no calibration
dataset required -- quantizes weights statically, activations
dynamically at inference time based on observed ranges).

Why dynamic over static quantization here: static quantization needs a
representative calibration dataset to pre-compute activation ranges,
which gives slightly better accuracy/latency but adds a real pipeline
step (running calibration images through the model first). Dynamic
quantization is a reasonable, simpler first cut for establishing the
size/accuracy tradeoff -- if the accuracy delta (checked below) turns
out to be unacceptable, static quantization with real MVTec calibration
images is the natural next experiment, not something to reach for
by default.

This script does NOT just quantize and declare victory -- it re-runs
the same PyTorch-vs-ONNX verification from export_onnx.py, but now
comparing INT8 ONNX output against the original fp32 PyTorch output,
since quantization is expected to introduce real (not just floating-point
rounding) numerical differences, and we need to know how large those
differences actually are before trusting this for anomaly scoring.

Usage:
    !python scripts/quantize_onnx.py --onnx-dir {DRIVE_ROOT}/onnx
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import torch

from core.feature_extractor import PatchFeatureExtractor


def quantize_and_verify(onnx_dir: Path, input_size: int = 224) -> dict:
    fp32_path = onnx_dir / "feature_extractor.onnx"
    int8_path = onnx_dir / "feature_extractor_int8.onnx"

    if not fp32_path.exists():
        raise FileNotFoundError(
            f"No fp32 ONNX model found at {fp32_path} -- run "
            f"scripts/export_onnx.py first."
        )

    from onnxruntime.quantization import quantize_dynamic, QuantType

    print(f"Quantizing {fp32_path} -> {int8_path}...")
    quantize_dynamic(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        weight_type=QuantType.QInt8,
    )

    # Size comparison -- include external-data companion files, since
    # (as we confirmed with the fp32 export) the .onnx file alone is not
    # the real size; large weight tensors live in a sidecar .data file.
    def total_size_mb(onnx_path: Path) -> float:
        total_bytes = onnx_path.stat().st_size
        data_path = onnx_path.with_suffix(onnx_path.suffix + ".data")
        if data_path.exists():
            total_bytes += data_path.stat().st_size
        return total_bytes / 1e6

    fp32_size = total_size_mb(fp32_path)
    int8_size = total_size_mb(int8_path)
    print(f"fp32 total size (graph + external data): {fp32_size:.1f} MB")
    print(f"int8 total size (graph + external data): {int8_size:.1f} MB")
    print(f"Size reduction: {fp32_size / int8_size:.2f}x")

    # Accuracy verification: compare INT8 ONNX output against the
    # ORIGINAL PyTorch fp32 model, not against the fp32 ONNX export --
    # we want to know the real end-to-end drift from "what the model
    # originally computed" to "what we'll actually run at inference",
    # not just the quantization step in isolation.
    import onnxruntime as ort

    model = PatchFeatureExtractor()
    model.eval()
    dummy_input = torch.randn(1, 3, input_size, input_size)

    with torch.no_grad():
        torch_output = model(dummy_input).numpy()

    session = ort.InferenceSession(str(int8_path))
    int8_output = session.run(
        ["patch_features"], {"image": dummy_input.numpy()}
    )[0]

    if int8_output.shape != torch_output.shape:
        raise RuntimeError(
            f"Shape mismatch after quantization: INT8 {int8_output.shape} vs "
            f"PyTorch {torch_output.shape}. Quantization broke the graph "
            f"structure -- do not trust this model."
        )

    max_abs_diff = float(np.abs(int8_output - torch_output).max())
    mean_abs_diff = float(np.abs(int8_output - torch_output).mean())
    # Relative difference matters more than absolute here, since these
    # are patch feature activations, not normalized probabilities --
    # their natural scale varies across channels, so a fixed absolute
    # tolerance is less meaningful than for a bounded output.
    relative_diff = float(
        np.abs(int8_output - torch_output).mean()
        / (np.abs(torch_output).mean() + 1e-8)
    )

    print(f"Max absolute difference vs original PyTorch fp32: {max_abs_diff:.4f}")
    print(f"Mean absolute difference: {mean_abs_diff:.4f}")
    print(f"Mean relative difference: {relative_diff:.2%}")
    print(
        "\nNOTE: this is a single random input, not a real accuracy "
        "measurement. A meaningful accuracy check means re-running "
        "scripts/run_eval.py's image/pixel AUROC and PRO metrics with "
        "the memory bank fed INT8-extracted features instead of fp32 "
        "ones, and comparing against the Phase 2 numbers already on "
        "record. Size/latency numbers above are trustworthy on their "
        "own; accuracy impact is NOT confirmed until that comparison "
        "is run."
    )

    return {
        "fp32_size_mb": fp32_size,
        "int8_size_mb": int8_size,
        "size_reduction_factor": fp32_size / int8_size,
        "max_abs_diff": max_abs_diff,
        "mean_abs_diff": mean_abs_diff,
        "mean_relative_diff": relative_diff,
    }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--onnx-dir", type=Path, required=True)
    parser.add_argument("--input-size", type=int, default=224)
    args = parser.parse_args()
    from pathlib import Path

    results = quantize_and_verify(Path(f"{DRIVE_ROOT}/onnx") ,224)
    print("\n=== Summary ===")
    for k, v in results.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()

In [ ]:
import sys
sys.path.insert(0, "/content/nightfall")

from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort
from sklearn.metrics import roc_auc_score

from core.patchcore import PatchCore
from eval.harness import EvalHarness
from eval.dataloader import load_category_test_data
from eval.train import ALL_MVTEC_CATEGORIES, checkpoint_path

ONNX_DIR = Path(f"{DRIVE_ROOT}/onnx")
int8_path = ONNX_DIR / "feature_extractor_int8.onnx"
CHECKPOINT_DIR = Path(f"{DRIVE_ROOT}/checkpoints")

class OnnxFeatureExtractor:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(str(onnx_path))
        self._input_name = self.session.get_inputs()[0].name
        self._output_name = self.session.get_outputs()[0].name

    def extract_patch_vectors(self, x):
        x_np = x.detach().cpu().numpy().astype(np.float32)
        fmap_np = self.session.run([self._output_name], {self._input_name: x_np})[0]
        fmap = torch.from_numpy(fmap_np)
        b, c, h, w = fmap.shape
        patch_vectors = fmap.permute(0, 2, 3, 1).reshape(b * h * w, c)
        return patch_vectors, (b, h, w)


import torch.nn.functional as F

class RawScoringMemoryBank:
    """
    Standalone reimplementation of MemoryBank.score(), WITHOUT the
    reweighting step -- defined directly here so this test doesn't
    depend on core/memory_bank.py already having the use_reweighting
    fix applied in your cloned repo.
    """
    def __init__(self, gaussian_blur_sigma=4.0, gaussian_blur_kernel=9):
        self.bank = None
        self.gaussian_blur_sigma = gaussian_blur_sigma
        self.gaussian_blur_kernel = gaussian_blur_kernel

    def fit(self, coreset_features):
        self.bank = coreset_features.clone()

    def _gaussian_kernel(self, device):
        k, sigma = self.gaussian_blur_kernel, self.gaussian_blur_sigma
        coords = torch.arange(k, dtype=torch.float32, device=device) - (k - 1) / 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g = g / g.sum()
        return (g[:, None] @ g[None, :]).view(1, 1, k, k)

    @torch.no_grad()
    def score(self, patch_features, spatial_shape, image_hw):
        b, h, w = spatial_shape
        device = patch_features.device
        bank = self.bank.to(device)

        dists = torch.cdist(patch_features, bank)
        nn_dists = dists.min(dim=1).values  # RAW distances, no reweighting

        scored = nn_dists.view(b, h * w)
        raw_dists = nn_dists.view(b, h, w)
        image_score = scored.max(dim=1).values

        pixel_map = raw_dists.unsqueeze(1)
        pixel_map = F.interpolate(pixel_map, size=image_hw, mode="bilinear", align_corners=False)
        kernel = self._gaussian_kernel(device)
        pad = self.gaussian_blur_kernel // 2
        pixel_map = F.conv2d(pixel_map, kernel, padding=pad)
        pixel_map = pixel_map.squeeze(1)

        class Result:
            pass
        r = Result()
        r.image_score = image_score
        r.pixel_map = pixel_map
        return r


model = PatchCore()
model.extractor = OnnxFeatureExtractor(int8_path)

fp32_reference = {
    "bottle": 0.997, "cable": 0.927, "capsule": 0.926, "carpet": 0.941,
    "grid": 0.799, "hazelnut": 0.956, "leather": 1.000, "metal_nut": 0.980,
    "pill": 0.810, "screw": 0.876, "tile": 0.993, "toothbrush": 0.989,
    "transistor": 0.969, "wood": 0.954, "zipper": 0.952,
}

print(f"[Real accuracy validation, INT8 + RAW scoring (guaranteed, self-contained)]")
results = {}
for category in ALL_MVTEC_CATEGORIES:
    ckpt_path = checkpoint_path(CHECKPOINT_DIR, category)
    if not ckpt_path.exists():
        print(f"[{category}] SKIPPED -- no checkpoint found")
        continue

    bank = RawScoringMemoryBank()
    bank.fit(torch.load(ckpt_path))

    try:
        test_data = load_category_test_data(Path(MVTEC_DIR), category, model.preprocessor)
    except FileNotFoundError as e:
        print(f"[{category}] SKIPPED -- {e}")
        continue

    patch_vectors, (b, h, w) = model.extractor.extract_patch_vectors(test_data.images)
    image_hw = test_data.images.shape[-2:]
    scoring = bank.score(patch_vectors, spatial_shape=(b, h, w), image_hw=tuple(image_hw))

    image_scores = scoring.image_score.numpy()
    auroc = roc_auc_score(test_data.image_labels, image_scores)
    results[category] = auroc

    fp32_auroc = fp32_reference.get(category)
    delta = f"{auroc - fp32_auroc:+.4f}" if fp32_auroc is not None else "n/a"
    print(f"[{category}] int8_raw_auroc={auroc:.4f}  fp32_reweighted={fp32_auroc}  delta={delta}")

if results:
    mean_int8 = sum(results.values()) / len(results)
    mean_fp32 = sum(fp32_reference[c] for c in results) / len(results)
    print(f"\n=== Summary ===")
    print(f"INT8 raw mean AUROC: {mean_int8:.4f}")
    print(f"fp32 reweighted mean AUROC: {mean_fp32:.4f}")
    print(f"Delta: {mean_int8 - mean_fp32:+.4f}")

In [ ]:
import sys
sys.path.insert(0, "/content/nightfall")

from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort

from core.patchcore import PatchCore
from core.memory_bank import MemoryBank
from eval.dataloader import load_category_test_data
from eval.train import checkpoint_path

ONNX_DIR = Path(f"{DRIVE_ROOT}/onnx")
int8_path = ONNX_DIR / "feature_extractor_int8.onnx"
CHECKPOINT_DIR = Path(f"{DRIVE_ROOT}/checkpoints")

class OnnxFeatureExtractor:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(str(onnx_path))
        self._input_name = self.session.get_inputs()[0].name
        self._output_name = self.session.get_outputs()[0].name

    def extract_patch_vectors(self, x):
        x_np = x.detach().cpu().numpy().astype(np.float32)
        fmap_np = self.session.run([self._output_name], {self._input_name: x_np})[0]
        fmap = torch.from_numpy(fmap_np)
        b, c, h, w = fmap.shape
        patch_vectors = fmap.permute(0, 2, 3, 1).reshape(b * h * w, c)
        return patch_vectors, (b, h, w)

model = PatchCore()
model.extractor = OnnxFeatureExtractor(int8_path)

bank = MemoryBank(model.bank_config)
bank.fit(torch.load(checkpoint_path(CHECKPOINT_DIR, "bottle")))
model.banks["bottle"] = bank

test_data = load_category_test_data(Path(MVTEC_DIR), "bottle", model.preprocessor)

# Manually replicate MemoryBank.score()'s internals so we can inspect
# raw vs weighted distances separately, rather than only seeing the
# final image_score.
patch_vectors, (b, h, w) = model.extractor.extract_patch_vectors(test_data.images)
bank_tensor = bank.bank

dists = torch.cdist(patch_vectors, bank_tensor)
nn_dists = dists.min(dim=1).values

topk = dists.topk(bank.config.reweight_k, dim=1, largest=False).values
softmax_weights = torch.softmax(-topk, dim=1)
weight = softmax_weights[:, 0]
weighted_dists = weight * nn_dists

raw_image_score = nn_dists.view(b, h * w).max(dim=1).values
weighted_image_score = weighted_dists.view(b, h * w).max(dim=1).values

from sklearn.metrics import roc_auc_score
labels = test_data.image_labels

raw_auroc = roc_auc_score(labels, raw_image_score.numpy())
weighted_auroc = roc_auc_score(labels, weighted_image_score.numpy())

print(f"RAW (unweighted) image AUROC:      {raw_auroc:.4f}")
print(f"REWEIGHTED (current) image AUROC:  {weighted_auroc:.4f}")
print(f"\nWeight stats: min={weight.min():.4f} max={weight.max():.4f} mean={weight.mean():.4f} std={weight.std():.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# topk, labels already available from the previous diagnostic cell
# Try a range of temperatures and see which one best restores AUROC
temperatures = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64.0, 128.0]

for temp in temperatures:
    scaled_weight = torch.softmax(-topk / temp, dim=1)[:, 0]
    scaled_weighted_dists = scaled_weight * nn_dists
    scaled_image_score = scaled_weighted_dists.view(b, h * w).max(dim=1).values
    auroc = roc_auc_score(labels, scaled_image_score.numpy())
    weight_std = scaled_weight.std().item()
    print(f"temp={temp:5.2f}  AUROC={auroc:.4f}  weight_std={weight_std:.4f}")

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import torch

# ---------------- FP32 model ----------------

fp32_model = PatchCore()

fp32_bank = MemoryBank(fp32_model.bank_config)
fp32_bank.fit(torch.load(checkpoint_path(CHECKPOINT_DIR, "bottle")))
fp32_model.banks["bottle"] = fp32_bank

patch_vectors_fp32, _ = fp32_model.extractor.extract_patch_vectors(test_data.images)

dists_fp32 = torch.cdist(patch_vectors_fp32, fp32_bank.bank)

# ---------------- INT8 model ----------------

patch_vectors_int8, _ = model.extractor.extract_patch_vectors(test_data.images)

dists_int8 = torch.cdist(patch_vectors_int8, bank.bank)

k = bank.config.reweight_k

topk_fp32_vals, topk_fp32_idx = dists_fp32.topk(k, largest=False)
topk_int8_vals, topk_int8_idx = dists_int8.topk(k, largest=False)

# ---------------------------------------------------
# 1. Top-1 nearest neighbour preservation
# ---------------------------------------------------

top1_preservation = (
    topk_fp32_idx[:, 0] == topk_int8_idx[:, 0]
).float().mean().item()

# ---------------------------------------------------
# 2. Top-k overlap
# ---------------------------------------------------

overlap = []

for a, b in zip(topk_fp32_idx, topk_int8_idx):
    overlap.append(
        len(set(a.tolist()) & set(b.tolist())) / k
    )

topk_overlap = np.mean(overlap)

# ---------------------------------------------------
# 3. Distance correlation
# ---------------------------------------------------

corrs = []

for fp, q in zip(topk_fp32_vals.numpy(), topk_int8_vals.numpy()):
    c, _ = spearmanr(fp, q)
    if not np.isnan(c):
        corrs.append(c)

mean_corr = np.mean(corrs)

# ---------------------------------------------------
# 4. Margin comparison
# ---------------------------------------------------

margin_fp32 = (
    topk_fp32_vals[:,1] - topk_fp32_vals[:,0]
).numpy()

margin_int8 = (
    topk_int8_vals[:,1] - topk_int8_vals[:,0]
).numpy()

print("="*60)
print("Quantization Geometry Analysis")
print("="*60)
print(f"Top-1 preservation : {100*top1_preservation:.2f}%")
print(f"Top-{k} overlap      : {100*topk_overlap:.2f}%")
print(f"Spearman corr.      : {mean_corr:.4f}")
print()
print(f"FP32 margin mean    : {margin_fp32.mean():.6f}")
print(f"INT8 margin mean    : {margin_int8.mean():.6f}")
print(f"FP32 margin std     : {margin_fp32.std():.6f}")
print(f"INT8 margin std     : {margin_int8.std():.6f}")
print()
print(f"Mean margin change  : {np.mean(np.abs(margin_fp32-margin_int8)):.6f}")

In [ ]:
import sys
sys.path.insert(0, "/content/nightfall")

from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort
from sklearn.metrics import roc_auc_score

from core.patchcore import PatchCore
from eval.harness import EvalHarness
from eval.dataloader import load_category_test_data
from eval.train import ALL_MVTEC_CATEGORIES, checkpoint_path

ONNX_DIR = Path(f"{DRIVE_ROOT}/onnx")
int8_path = ONNX_DIR / "feature_extractor_int8.onnx"
CHECKPOINT_DIR = Path(f"{DRIVE_ROOT}/checkpoints")

class OnnxFeatureExtractor:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(str(onnx_path))
        self._input_name = self.session.get_inputs()[0].name
        self._output_name = self.session.get_outputs()[0].name

    def extract_patch_vectors(self, x):
        x_np = x.detach().cpu().numpy().astype(np.float32)
        fmap_np = self.session.run([self._output_name], {self._input_name: x_np})[0]
        fmap = torch.from_numpy(fmap_np)
        b, c, h, w = fmap.shape
        patch_vectors = fmap.permute(0, 2, 3, 1).reshape(b * h * w, c)
        return patch_vectors, (b, h, w)


import torch.nn.functional as F

class RawScoringMemoryBank:
    """
    Standalone reimplementation of MemoryBank.score(), WITHOUT the
    reweighting step -- defined directly here so this test doesn't
    depend on core/memory_bank.py already having the use_reweighting
    fix applied in your cloned repo.
    """
    def __init__(self, gaussian_blur_sigma=4.0, gaussian_blur_kernel=9):
        self.bank = None
        self.gaussian_blur_sigma = gaussian_blur_sigma
        self.gaussian_blur_kernel = gaussian_blur_kernel

    def fit(self, coreset_features):
        self.bank = coreset_features.clone()

    def _gaussian_kernel(self, device):
        k, sigma = self.gaussian_blur_kernel, self.gaussian_blur_sigma
        coords = torch.arange(k, dtype=torch.float32, device=device) - (k - 1) / 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g = g / g.sum()
        return (g[:, None] @ g[None, :]).view(1, 1, k, k)

    @torch.no_grad()
    def score(self, patch_features, spatial_shape, image_hw):
        b, h, w = spatial_shape
        device = patch_features.device
        bank = self.bank.to(device)

        dists = torch.cdist(patch_features, bank)
        nn_dists = dists.min(dim=1).values  # RAW distances, no reweighting

        scored = nn_dists.view(b, h * w)
        raw_dists = nn_dists.view(b, h, w)
        image_score = scored.max(dim=1).values

        pixel_map = raw_dists.unsqueeze(1)
        pixel_map = F.interpolate(pixel_map, size=image_hw, mode="bilinear", align_corners=False)
        kernel = self._gaussian_kernel(device)
        pad = self.gaussian_blur_kernel // 2
        pixel_map = F.conv2d(pixel_map, kernel, padding=pad)
        pixel_map = pixel_map.squeeze(1)

        class Result:
            pass
        r = Result()
        r.image_score = image_score
        r.pixel_map = pixel_map
        return r


model = PatchCore()
model.extractor = OnnxFeatureExtractor(int8_path)

fp32_reference = {
    "bottle": 0.997, "cable": 0.927, "capsule": 0.926, "carpet": 0.941,
    "grid": 0.799, "hazelnut": 0.956, "leather": 1.000, "metal_nut": 0.980,
    "pill": 0.810, "screw": 0.876, "tile": 0.993, "toothbrush": 0.989,
    "transistor": 0.969, "wood": 0.954, "zipper": 0.952,
}

print(f"[Real accuracy validation, INT8 + RAW scoring (guaranteed, self-contained)]")
results = {}
for category in ALL_MVTEC_CATEGORIES:
    ckpt_path = checkpoint_path(CHECKPOINT_DIR, category)
    if not ckpt_path.exists():
        print(f"[{category}] SKIPPED -- no checkpoint found")
        continue

    bank = RawScoringMemoryBank()
    bank.fit(torch.load(ckpt_path))

    try:
        test_data = load_category_test_data(Path(MVTEC_DIR), category, model.preprocessor)
    except FileNotFoundError as e:
        print(f"[{category}] SKIPPED -- {e}")
        continue

    patch_vectors, (b, h, w) = model.extractor.extract_patch_vectors(test_data.images)
    image_hw = test_data.images.shape[-2:]
    scoring = bank.score(patch_vectors, spatial_shape=(b, h, w), image_hw=tuple(image_hw))

    image_scores = scoring.image_score.numpy()
    auroc = roc_auc_score(test_data.image_labels, image_scores)
    results[category] = auroc

    fp32_auroc = fp32_reference.get(category)
    delta = f"{auroc - fp32_auroc:+.4f}" if fp32_auroc is not None else "n/a"
    print(f"[{category}] int8_raw_auroc={auroc:.4f}  fp32_reweighted={fp32_auroc}  delta={delta}")

if results:
    mean_int8 = sum(results.values()) / len(results)
    mean_fp32 = sum(fp32_reference[c] for c in results) / len(results)
    print(f"\n=== Summary ===")
    print(f"INT8 raw mean AUROC: {mean_int8:.4f}")
    print(f"fp32 reweighted mean AUROC: {mean_fp32:.4f}")
    print(f"Delta: {mean_int8 - mean_fp32:+.4f}")

In [ ]:
patch_vectors, _ = model.extractor.extract_patch_vectors(train_data.images)
int8_bank = MemoryBank(model.bank_config)
int8_bank.fit(patch_vectors)
model.banks["bottle"] = int8_bank

In [ ]:
"""
Loads MVTec AD test-set images and ground-truth masks into the
CategoryTestData shape EvalHarness expects.

Directory convention (see train_all_categories.py's docstring):
    <data_root>/<category>/test/<defect_type>/*.png
    <data_root>/<category>/ground_truth/<defect_type>/*_mask.png

"good" test images have no ground-truth mask (there's no defect to
annotate) -- we fill in an all-zero mask for those rather than skipping
them, since pixel AUROC and PRO both need every pixel across the full
test set, including the true-negative pixels that only "good" images
contribute.

Mask/image pairing is by matching numeric index within a defect-type
folder (e.g. test/broken_large/001.png <-> ground_truth/broken_large/001_mask.png),
per the same convention train_all_categories.py's HuggingFace fallback
writes. This was spot-checked manually against several real image/mask
pairs before relying on it here (see project notes) -- the pairing is
NOT verified programmatically at load time, so a future change to
either side's naming convention could silently break this; a mismatch
would corrupt every downstream metric without raising an error, since
shapes would still line up even if content doesn't correspond.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from PIL import Image

from core.preprocessing import ImagePreprocessor
from eval.harness import CategoryTestData


def load_category_train_data(
    data_root: Path,
    category: str,
    preprocessor: ImagePreprocessor,
) -> CategoryTestData:
    """
    Walks <data_root>/<category>/test/ and .../ground_truth/, builds a
    CategoryTestData with preprocessed images, image-level labels
    (0=good, 1=defective), and pixel-level binary masks matched to the
    preprocessor's output resolution (crop_size x crop_size).
    """
    category_root = data_root / category
    test_root = category_root / "train/good"
    gt_root = category_root / "ground_truth"

    if not test_root.exists():
        raise FileNotFoundError(f"No test/ directory found at {test_root}")

    crop_size = preprocessor.config.crop_size

    image_paths: list[Path] = []
    image_labels: list[int] = []
    mask_arrays: list[np.ndarray] = []

    for defect_dir in sorted(test_root.iterdir()):
        if not defect_dir.is_dir():
            continue
        defect_type = defect_dir.name
        is_good = defect_type == "good"

        for img_path in sorted(defect_dir.glob("*.png")):
            image_paths.append(img_path)
            image_labels.append(0 if is_good else 1)

            # Match by numeric index within this defect type, per the
            # convention: 001.png <-> 001_mask.png.
            idx = img_path.stem  # e.g. "001"
            mask_path = gt_root / defect_type / f"{idx}_mask.png"
            if not mask_path.exists():
                raise FileNotFoundError(
                    f"Expected mask at {mask_path} for test image {img_path}, "
                    f"but it doesn't exist. Pairing assumption may be broken "
                    f"for this category/defect_type -- do not silently skip, "
                    f"since that would misalign image_labels and pixel_labels "
                    f"arrays with the images tensor."
                )

            mask_img = Image.open(mask_path).convert("L")
            # Resize mask to match the preprocessed image's crop_size --
            # nearest-neighbor, not bilinear, since this is a binary mask
            # and interpolating would introduce fractional "defect-ish"
            # pixel values that don't correspond to a real annotation.
            mask_img = mask_img.resize((crop_size, crop_size), Image.NEAREST)
            mask_arr = (np.array(mask_img) > 127).astype(np.uint8)
            mask_arrays.append(mask_arr)

    if not image_paths:
        raise FileNotFoundError(f"No test images found under {test_root}")

    images = preprocessor.batch(image_paths)
    labels = np.array(image_labels, dtype=np.int64)
    masks = np.stack(mask_arrays, axis=0)

    return CategoryTestData(
        category=category,
        images=images,
        image_labels=labels,
        pixel_labels=masks,
    )
train_data = load_category_train_data(Path(MVTEC_DIR), category, model.preprocessor)

In [ ]:
from pathlib import Path

category = "bottle"

data_root = Path("/content/drive/MyDrive/Nightfall/data/mvtec_ad")

train_dir = data_root / category / "train" / "good"
image_paths = sorted(train_dir.glob("*.png"))

print(train_dir)
print(f"Found {len(image_paths)} training images")

model.fit_from_paths(category, image_paths)

int8_bank = model.banks[category]

print("INT8 bank shape:", int8_bank.bank.shape)

In [ ]:
import sys
sys.path.insert(0, "/content/nightfall")

from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort
from sklearn.metrics import roc_auc_score

from core.patchcore import PatchCore
from core.coreset import GreedyCoresetSampler
from eval.dataloader import load_category_test_data

ONNX_DIR = Path(f"{DRIVE_ROOT}/onnx")
int8_path = ONNX_DIR / "feature_extractor_int8.onnx"
MVTEC_PATH = Path(MVTEC_DIR)

class OnnxFeatureExtractor:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(str(onnx_path))
        self._input_name = self.session.get_inputs()[0].name
        self._output_name = self.session.get_outputs()[0].name

    def extract_patch_vectors(self, x):
        x_np = x.detach().cpu().numpy().astype(np.float32)
        fmap_np = self.session.run([self._output_name], {self._input_name: x_np})[0]
        fmap = torch.from_numpy(fmap_np)
        b, c, h, w = fmap.shape
        patch_vectors = fmap.permute(0, 2, 3, 1).reshape(b * h * w, c)
        return patch_vectors, (b, h, w)


model = PatchCore()
model.extractor = OnnxFeatureExtractor(int8_path)
coreset_sampler = GreedyCoresetSampler()

fp32_reweighted = {"hazelnut": 0.956, "metal_nut": 0.980}
int8_raw_mismatched = {"hazelnut": 0.4864, "metal_nut": 0.6232}

print("[Refit test: INT8-extracted TRAIN features -> new memory bank -> INT8-extracted TEST scoring]")
for category in ["metal_nut"]:
    train_dir = MVTEC_PATH / category / "train" / "good"
    train_paths = sorted(train_dir.glob("*.png"))
    print(f"\n[{category}] refitting bank from {len(train_paths)} training images using INT8 extractor...")

    train_images = model.preprocessor.batch(train_paths)
    patch_vectors, _shape = model.extractor.extract_patch_vectors(train_images)
    coreset = coreset_sampler.select(patch_vectors)
    print(f"[{category}] new INT8-fitted bank size: {coreset.shape[0]}")

    test_data = load_category_test_data(MVTEC_PATH, category, model.preprocessor)
    test_patch_vectors, (b, h, w) = model.extractor.extract_patch_vectors(test_data.images)

    dists = torch.cdist(test_patch_vectors, coreset)
    nn_dists = dists.min(dim=1).values
    image_score = nn_dists.view(b, h * w).max(dim=1).values

    auroc = roc_auc_score(test_data.image_labels, image_score.numpy())

    print(f"[{category}] fp32 reweighted (original):       {fp32_reweighted[category]:.4f}")
    print(f"[{category}] INT8 raw, MISMATCHED bank:         {int8_raw_mismatched[category]:.4f}")
    print(f"[{category}] INT8 raw, REFIT (consistent) bank: {auroc:.4f}")

# Testing the AUC of 1.0

In [ ]:
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score

from core.patchcore import PatchCore
from core.coreset import GreedyCoresetSampler
from eval.dataloader import load_category_test_data
import onnxruntime as ort

ONNX_DIR = Path(f"{DRIVE_ROOT}/onnx")
int8_path = ONNX_DIR / "feature_extractor_int8.onnx"
MVTEC_PATH = Path(MVTEC_DIR)


class OnnxFeatureExtractor:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        self._input_name = self.session.get_inputs()[0].name
        self._output_name = self.session.get_outputs()[0].name

    def extract_patch_vectors(self, x, batch_size=16):
        # Batched to avoid holding the whole set's activations in memory
        # at once. Last batch may be smaller than batch_size -- Python
        # slicing handles this safely (yields whatever remains), and
        # each batch's own (b, h, w) is used only for its own reshape,
        # never assumed constant across batches.
        vectors_list = []
        for i in range(0, x.shape[0], batch_size):
            batch = x[i:i + batch_size].numpy().astype(np.float32)
            fmap = self.session.run([self._output_name], {self._input_name: batch})[0]
            fmap = torch.from_numpy(fmap)
            b, c, h, w = fmap.shape
            vectors_list.append(fmap.permute(0, 2, 3, 1).reshape(b * h * w, c))
        return torch.cat(vectors_list, dim=0)


def investigate(category):
    model_extractor = OnnxFeatureExtractor(int8_path)
    preprocessor = PatchCore().preprocessor
    coreset_sampler = GreedyCoresetSampler()

    train_dir = MVTEC_PATH / category / "train" / "good"
    train_paths = sorted(train_dir.glob("*.png"))
    train_images = preprocessor.batch(train_paths)
    patch_vectors = model_extractor.extract_patch_vectors(train_images)
    coreset = coreset_sampler.select(patch_vectors)

    test_data = load_category_test_data(MVTEC_PATH, category, preprocessor)
    test_vectors = model_extractor.extract_patch_vectors(test_data.images)

    # Assumes every image produces the same number of patches, which
    # holds here because ImagePreprocessor always resizes/crops to a
    # fixed 224x224 -- stated explicitly rather than left implicit.
    n_test_images = test_data.images.shape[0]
    patches_per_image = test_vectors.shape[0] // n_test_images

    image_scores = []
    for i in range(n_test_images):
        img_patches = test_vectors[i * patches_per_image : (i + 1) * patches_per_image]
        dists = torch.cdist(img_patches, coreset)
        image_scores.append(dists.min(dim=1).values.max().item())
    image_scores = np.array(image_scores)

    labels = test_data.image_labels
    normal = np.sort(image_scores[labels == 0])
    defect = np.sort(image_scores[labels == 1])

    auroc = roc_auc_score(labels, image_scores)

    print(f"\n=== {category} ===")
    print(f"AUROC: {auroc:.4f}")
    print(f"Normal scores (n={len(normal)}), sorted: {np.round(normal, 4)}")
    print(f"Defect scores (n={len(defect)}), sorted: {np.round(defect, 4)}")
    print(f"Margin (defect_min - normal_max): {defect.min() - normal.max():.4f}")
    print(f"Score range (all): {image_scores.min():.4f} to {image_scores.max():.4f}")
    print(f"Margin as % of score range: {100*(defect.min()-normal.max())/(image_scores.max()-image_scores.min()):.1f}%")


investigate("hazelnut")
investigate("metal_nut")

# Benchmarking

In [ ]:
!python scripts/benchmark_latency.py --onnx-dir {DRIVE_ROOT}/onnx

In [ ]:
!python scripts/generate_test_image_header.py \
    --image-path {MVTEC_DIR}/bottle/test/broken_large/001.png \
    --output serving/test_image.h

In [ ]:
from google.colab import files
files.download("serving/test_image.h")

# Start Server

In [ ]:
!pip install grpcio grpcio-tools --quiet

In [ ]:
import subprocess, time

nightfall_grpc_proc = subprocess.Popen([
    "python", "serving/run_grpc_server.py",
    "--checkpoint-dir", f"{DRIVE_ROOT}/checkpoints",
    "--thresholds-path", f"{DRIVE_ROOT}/checkpoints/thresholds.json",
    "--port", "50051"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(15)
print("gRPC poll:", nightfall_grpc_proc.poll())

In [ ]:
nightfall_gateway_proc = subprocess.Popen([
    "python", "serving/rest_gateway.py",
    "--grpc-target", "localhost:50051",
    "--port", "8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)
print("REST gateway started successfully")

In [ ]:
!python scripts/calibrate_thresholds.py \
    --data-root {MVTEC_DIR} \
    --checkpoint-dir {DRIVE_ROOT}/checkpoints \
    --output {DRIVE_ROOT}/checkpoints/thresholds.json

In [ ]:
# Testing with an actual payload
import subprocess, time

proc = subprocess.Popen([
    "python", "serving/run_grpc_server.py",
    "--checkpoint-dir", f"{DRIVE_ROOT}/checkpoints",
    "--thresholds-path", f"{DRIVE_ROOT}/checkpoints/thresholds.json",
    "--port", "50051"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(10)

if proc.poll() is not None:
    print("Server crashed. Output:")
    print(proc.stdout.read())
else:
    print("Server running, testing a real request...")

    import sys
    sys.path.insert(0, "/content/Nightfall/serving")
    import grpc
    import nightfall_pb2
    import nightfall_pb2_grpc

    channel = grpc.insecure_channel("localhost:50051")
    stub = nightfall_pb2_grpc.NightfallInferenceStub(channel)

    # Test 1: ListCategories
    categories_response = stub.ListCategories(nightfall_pb2.ListCategoriesRequest())
    print(f"Available categories: {list(categories_response.categories)}")

    # Test 2: a real bottle test image, actual defective one
    test_image_path = f"{MVTEC_DIR}/bottle/test/broken_large/001.png"
    with open(test_image_path, "rb") as f:
        image_bytes = f.read()

    request = nightfall_pb2.AnomalyRequest(
        category="bottle",
        image_data=image_bytes,
        image_format="png",
    )
    response = stub.DetectAnomaly(request)
    print(f"\nDefective image test:")
    print(f"  success: {response.success}")
    print(f"  anomaly_score: {response.anomaly_score}")
    print(f"  is_anomalous: {response.is_anomalous}")
    print(f"  heatmap size: {len(response.heatmap_png)} bytes")
    print(f"  server-side latency: {response.inference_latency_ms:.1f}ms")

    # Test 3: a real good/normal bottle image, should NOT be anomalous
    normal_image_path = f"{MVTEC_DIR}/bottle/test/good/001.png"
    with open(normal_image_path, "rb") as f:
        normal_bytes = f.read()

    normal_request = nightfall_pb2.AnomalyRequest(
        category="bottle", image_data=normal_bytes, image_format="png",
    )
    normal_response = stub.DetectAnomaly(normal_request)
    print(f"\nNormal image test:")
    print(f"  anomaly_score: {normal_response.anomaly_score}")
    print(f"  is_anomalous: {normal_response.is_anomalous}")

    # Test 4: invalid category, should fail cleanly
    bad_request = nightfall_pb2.AnomalyRequest(
        category="not_a_real_category", image_data=image_bytes, image_format="png",
    )
    bad_response = stub.DetectAnomaly(bad_request)
    print(f"\nInvalid category test:")
    print(f"  success: {bad_response.success}")
    print(f"  error_message: {bad_response.error_message}")

    proc.terminate()

# Localtunnel setup

In [ ]:
 #--- Cell A: install localtunnel ---
!npm install -g localtunnel


In [ ]:
# --- Cell B: start the tunnel as a background process, capture its output ---
import subprocess, time, re

localtunnel_proc = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
start_time = time.time()
output_lines = []

while time.time() - start_time < 20:
    line = localtunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    output_lines.append(line)
    match = re.search(r"https?://[a-zA-Z0-9\-]+\.loca\.lt", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"Localtunnel URL: {public_url}")
    # localtunnel's default URL is https -- we specifically want the
    # plain http version to avoid the TLS handshake issue, so swap the
    # scheme explicitly rather than trust whatever it printed.
    http_url = public_url.replace("https://", "http://")
    print(f"Plain HTTP version to use in esp32_client.ino: {http_url}/detect")
else:
    print("Could not detect tunnel URL within 20s. Full output so far:")
    print("".join(output_lines))

In [ ]:

# --- Cell C: verify the PLAIN HTTP endpoint works before touching Wokwi ---
# Localtunnel also requires a one-time "click to continue" bypass header
# for its own abuse-prevention interstitial, similar to ngrok's -- pass
# it here too, just in case.
import requests

test_response = requests.get(
    f"{http_url}/categories",
    headers={"bypass-tunnel-reminder": "true"},
)
print(f"Status: {test_response.status_code}")
print(f"Response: {test_response.text}")


# --- Cell D (optional): stop the tunnel when done ---
# localtunnel_proc.terminate()
